In [1]:
#| default_exp latex.preamble

In [2]:
#| export
import re
from os import PathLike
from pathlib import Path
import warnings

In [3]:
from trouver.helper.files_and_folders import text_from_file
from trouver.helper.tests import _test_directory

## Divide the preamble from the rest of the document

## Divide the preamble from the rest of the document

Some macros and commands defined in the preamble seem to prevent the `pylatexenc` methods from properly identifying the document environment/node in a LaTeX document. To circumvent this, we define a function to divide the preamble from the rest of the document. Per N14 (decided 2026-10-30, option c), preamble-less input — e.g. `\input` chapter fragments — divides as body-only with an empty preamble, so multi-file papers' fragments become notes in their own right.

In [4]:
#| export

def divide_preamble(

        text: str, # LaTeX document

        document_environment_name: str = "document"

        ) -> tuple[str, str]:

    """Divide the preamble from the rest of a LaTeX document.

    If no `\begin{document}` is found (e.g. an `\input` chapter fragment), the whole
    `text` is returned as the body with an empty preamble (N14, option c).

    """

    begin_environment_str = rf'\begin{{{document_environment_name}}}'

    pattern = re.compile(re.escape(begin_environment_str))

    match = re.search(pattern, text)

    if match is None:

        return '', text

    start_match, end_match = match.span()

    return text[:start_match], text[start_match:]




<>:13: SyntaxWarning: invalid escape sequence '\i'
<>:13: SyntaxWarning: invalid escape sequence '\i'
C:\Users\hyunj\AppData\Local\Temp\ipykernel_2276\3289120566.py:13: SyntaxWarning: invalid escape sequence '\i'
  If no `\begin{document}` is found (e.g. an `\input` chapter fragment), the whole


In [5]:
latex_file_path = _test_directory() / 'latex_examples' / 'example_with_a_command_with_begin.tex'
text = text_from_file(latex_file_path)

preamble, document = divide_preamble(text)
assert r'\begin{displaymath}' in preamble
assert r'Hyun Jong Kim' in preamble

assert r'Hyun Jong Kim' not in document
assert document.startswith(r'\begin{document}')
assert document.endswith('\\end{document}')

In [6]:
from fastcore.test import test_eq
# n14t1 (N14, decided 2026-10-30 — option c): preamble-less input divides as body-only
fragment = r'\section{Intro}\nSome text.\n'
test_eq(divide_preamble(fragment), ('', fragment))

## Include `.sty` file's content into preamble

Writers often define custom commands in `.sty` files 

In [7]:
#| export
def replace_inclusion_of_style_file_with_code(
        document: str,
        dir: PathLike  # The directory containing the style file.
        ) -> str:  # The modified document with style file inclusions replaced by their contents.
    r"""
    Replace style file inclusions in `document` with the code of the style files.

    This function searches for occurrences of `\usepackage{...}`, `\input{...}`, 
    `\import{...}{...}`, `\includefrom{...}{...}`, and `\subincludefrom{...}{...}`
    and replaces them with the actual contents of the corresponding `.sty` or `.tex` files, if available.
    """
    def replace_style_file(match):
        command = match.group(1)  # Command type
        if command in ['import', 'includefrom', 'subincludefrom']:
            path = match.group(2)
            filename = match.group(3)
            file_path = Path(dir) / path / filename
        else:
            filename = match.group(2)
            file_path = Path(dir) / filename
        
        # Try different extensions if no extension is provided
        if not file_path.suffix:
            for ext in ['.sty', '.tex']:
                temp_path = file_path.with_suffix(ext)
                if temp_path.exists():
                    file_path = temp_path
                    break
        
        # Only process .sty and .tex files
        if file_path.suffix not in ['.sty', '.tex']:
            return match.group(0)
        
        try:
            # Read and return the contents of the file
            with open(file_path, 'r', encoding='utf-8') as file:
                return f"% Start of included file: {file_path.name}\n" + file.read() + f"\n% End of included file: {file_path.name}"
        except FileNotFoundError:
            warnings.warn(f"File {file_path} not found. Keeping original command.", UserWarning)
            return match.group(0)

    # Regex pattern to match all relevant commands for style files
    pattern = r'\\(usepackage|input)\{([^}]+)\}|\\(import|includefrom|subincludefrom)\{([^}]+)\}\{([^}]+)\}'
    
    # Replace all matches in the document
    return re.sub(pattern, replace_style_file, document)

The  `replace_inclusion_of_style_file_with_code` function substitutes the code in style files into the appropriate locations in a latex document.

In [8]:
latex_folder = _test_directory() / 'latex_examples' / 'latex_example_with_style_file'
main_file = latex_folder / 'main.tex'
with open(main_file, 'r', encoding='utf-8') as file:
    document = file.read()
output = replace_inclusion_of_style_file_with_code(document, latex_folder)
# preamble, body = divide_preamble(document)
# commands = custom_commands(preamble)
print(output)

\documentclass{article}

% Include a style file
% Start of included file: mystyle.sty
% My custom style definitions
\newcommand{\mystylecommand}[1]{\textbf{#1}}

% End of included file: mystyle.sty

\begin{document}

% Include a non-style file
% Start of included file: nonstylefile.tex
% This is a regular text file, not a style file.
This content will be included as plain text.

% End of included file: nonstylefile.tex


Hello, world! This document uses styles defined in mystyle.sty.

\end{document}



It may be best practice to apply `replace_inclusion_of_styl_file_with_code` to the preamble of the document assuming that the latex document only includes style files in the preamble.

In [9]:
latex_folder = _test_directory() / 'latex_examples' / 'latex_example_with_style_file'
main_file = latex_folder / 'main.tex'
with open(main_file, 'r', encoding='utf-8') as file:
    document = file.read()
preamble, body = divide_preamble(document)
output = replace_inclusion_of_style_file_with_code(preamble, latex_folder)
# commands = custom_commands(preamble)
print(output)

\documentclass{article}

% Include a style file
% Start of included file: mystyle.sty
% My custom style definitions
\newcommand{\mystylecommand}[1]{\textbf{#1}}

% End of included file: mystyle.sty


